In [1]:
print("kernel ok")

kernel ok


In [2]:
import os
import findspark

findspark.init('/home/ubuntu/spark')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Jupyter-Interactive-Analysis") \
    .master("spark://g23-master:7077") \
    .getOrCreate()

print("Spark is successfully connected")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 08:36:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark is successfully connected


In [3]:
df = spark.read.parquet("/data/cleaned_reddit/")
#all column names and data types.
df.printSchema()

root
 |-- author: string (nullable = true)
 |-- body: string (nullable = true)
 |-- content: string (nullable = true)
 |-- content_len: long (nullable = true)
 |-- id: string (nullable = true)
 |-- normalizedBody: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- summary_len: long (nullable = true)
 |-- title: string (nullable = true)
 |-- content_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [4]:
#First 3 rows of the data, with full content
df.show(3, truncate=True)

+------------------+--------------------+--------------------+-----------+-------+--------------------+---------+------------+--------------------+-----------+--------------------+--------------------+
|            author|                body|             content|content_len|     id|      normalizedBody|subreddit|subreddit_id|             summary|summary_len|               title|      content_tokens|
+------------------+--------------------+--------------------+-----------+-------+--------------------+---------+------------+--------------------+-----------+--------------------+--------------------+
|      chikinpickle|It sounds like he...|It sounds like he...|        122|can350z|It sounds like he...|      sex|    t5_2qh3p|dont worry about ...|          7|How can I tell if...|[sounds, like, he...|
|   GLaDOS_speaking|If you're interes...|If you're interes...|        319|can9xgt|If you're interes...|GameDeals|    t5_2qwx3|it's a great, pai...|         26|                NULL|[interested,

In [5]:
with open("swear_words.txt", "r", encoding="utf-8") as f:
    swear_words = [line.strip() for line in f if line.strip()]

In [6]:
import pyspark.sql.functions as F
swear_words_array = F.array([F.lit(word) for word in swear_words])

In [7]:
df = df.withColumn(
    "swear_word_count",
    F.size(F.array_intersect(F.col("content_tokens"), swear_words_array))
)

In [8]:
result = df.groupBy("subreddit") \
    .agg(
        F.avg("swear_word_count").alias("avg_swear_words_per_post"),
        F.count("*").alias("post_count")
    ) \
    .filter(F.col("post_count") > 1000) \
    .orderBy(F.col("avg_swear_words_per_post").desc())
result.show(10)

+-------------------+------------------------+----------+
|          subreddit|avg_swear_words_per_post|post_count|
+-------------------+------------------------+----------+
|               rant|       2.543175487465181|      2154|
|   fatpeoplestories|      2.4985789687373123|      4926|
|cripplingalcoholism|      1.9402472527472527|      1456|
|        breakingmom|      1.8722126929674099|      1166|
|         TheRedPill|      1.7532676452845366|      4973|
|             asktrp|      1.7159904534606205|      2095|
|  howtonotgiveafuck|                  1.6975|      1600|
|         offmychest|      1.6937991266375545|     17175|
|        LetsNotMeet|       1.607361963190184|      1793|
|            opiates|      1.5918218327856881|      2739|
+-------------------+------------------------+----------+
only showing top 10 rows



In [9]:
result = df.groupBy("subreddit") \
    .agg(
        F.avg("swear_word_count").alias("avg_swear_words_per_post"),
        F.count("*").alias("post_count")
    ) \
    .filter(F.col("post_count") > 1000) \
    .orderBy(F.col("avg_swear_words_per_post").asc())
result.show(10)

+--------------------+------------------------+----------+
|           subreddit|avg_swear_words_per_post|post_count|
+--------------------+------------------------+----------+
|minecraftsuggestions|    0.060218978102189784|      1096|
|          askscience|     0.06563097117221899|     12037|
|        feedthebeast|     0.09754562617998741|      1589|
|           argentina|     0.09818875119161107|      1049|
|                math|      0.1021671826625387|      1938|
|              sweden|     0.10320284697508897|      1124|
|    learnprogramming|     0.11371080954609265|      4274|
|  KerbalSpaceProgram|     0.11587982832618025|      2097|
|           applehelp|     0.11830357142857142|      1792|
|        ABraThatFits|     0.12023460410557185|      1023|
+--------------------+------------------------+----------+
only showing top 10 rows

